## Benchmarking multiple `Machine Learning` models

*Theme*

> **Every model has `Strengths` and `Weaknesses`**

From the previous recommendation i.e **Phase 4 should NOT end after training a single Logistic Regression model**. A real ML engineer's job is to answer:

> **"Which model is best for this problem, and why?"**

**Not**:

> **"How do I train a model?"**

*Note*: This introduces you to one of the most important principles in Machine Learning Engineering:

> **Benchmark → Compare → Select → Improve**

---

**Business scenario**

Suppose shell Nigeria wants to deploy an AI system that classifies incident reports. You train one model.

*Accuracy*:

```
    82%
```

Management asks

> "Is that good?"

Can you answer? `No`. Maybe another model achieves

```
    95%
```

You'll never know unless you `compare`. This is called **benchmarking**. Note the keyword *benchmarking*!

---

*A quick question*: Who remembers the project where we applied this technique of **benchmarking**?

---

**Learning objectives:**

* Why we compare models
* How different algorithms learn
* Bias vs Variance
* Speed vs Accuracy
* Interpretability vs Performance
* Model benchmarking
* Selecting the best model

---

### Update `project structure`

Instead of

```
    model.py
```

*We'll create;*

```
    src/

        models/

            logistic_regression.py

            naive_bayes.py

            svm_classifier.py

            decision_tree.py

            random_forest.py
```

Notice, each algorithm has its own class. This is `professional`.

---

**`Why split models?`**

Should ```model.py``` contain 500 lines for every algorithm? No. Each model has one responsibility. 

Exactly like

```
DataLoader

TextPreprocessor

FeatureEngineer
```
 
Now `models` also follow the `single responsibility` principle.

---

### New `folder`

```
src/

models/

│
├── __init__.py
├── logistic_regression.py
├── naive_bayes.py
├── svm_classifier.py
├── decision_tree.py
└── random_forest.py
```

---

### 1. Logistic Regression; 

```python
    from sklearn.linear_model import LogisticRegression


    class LogisticIncidentClassifier:

        def __init__(self):

            self.model = LogisticRegression(max_iter=1000)

        def train(self, X_train, y_train):

            self.model.fit(X_train, y_train)

        def predict(self, X):

            return self.model.predict(X)
```

Visit: https://www.github.com/devmab24/3Logy... to learn more

---

### 2. Naive Bayes

`Naive Bayes` assumes words are independent. This assumption is unrealistic, but surprisingly effective for text classification.

```python
    from sklearn.naive_bayes import MultinomialNB


    class NaiveBayesClassifier:

        def __init__(self):

            self.model = MultinomialNB()

        def train(self, X_train, y_train):

            self.model.fit(X_train, y_train)

        def predict(self, X):

            return self.model.predict(X)
```


**Why Naive Bayes?**

It is:

* fast
* memory efficient
* excellent on TF-IDF
* excellent on CountVectorizer

AWS exams love it.

Visit: https://www.github.com/devmab24/3Logy... to learn more

---

### 3. Support Vector Machine (SVM)

One of the best classical text classifiers.

```python
    from sklearn.svm import LinearSVC


    class SVMClassifier:

        def __init__(self):

            self.model = LinearSVC()

        def train(self, X_train, y_train):

            self.model.fit(X_train, y_train)

        def predict(self, X):

            return self.model.predict(X)
```

Linear `SVM` is frequently the strongest classical text classifier.

Visit: https://www.github.com/devmab24/3Logy... to learn more

---

### 4. Decision Tree

You already know trees from theory.

```python
    from sklearn.tree import DecisionTreeClassifier


    class DecisionTreeIncidentClassifier:

        def __init__(self):

            self.model = DecisionTreeClassifier(
                random_state=42
            )

        def train(self, X_train, y_train):

            self.model.fit(X_train, y_train)

        def predict(self, X):

            return self.model.predict(X)
```

It's very interpretable. But not always the best for sparse text vectors.

Visit: https://www.github.com/devmab24/3Logy... to learn more

---

### 5. Random Forest

```python
    from sklearn.ensemble import RandomForestClassifier


    class RandomForestIncidentClassifier:

        def __init__(self):

            self.model = RandomForestClassifier(
                random_state=42
            )

        def train(self, X_train, y_train):

            self.model.fit(X_train, y_train)

        def predict(self, X):

            return self.model.predict(X)
```

Many trees. Less overfitting. Usually slower.

Visit: https://www.github.com/devmab24/3Logy... to learn more

---

### Build `benchmark script`

Instead of ```train.py```, **create** ```benchmark.py``` module which will be used to `compare` every algorithm. Inside the `benchmark.py` module add:

```python
"""Multi-model benchmarking for SIRA."""

import logging
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from src.models.decision_tree import (DecisionTreeIncidentClassifier,)
from src.models.logistic_regression import (LogisticIncidentClassifier,)
from src.models.naive_bayes import (NaiveBayesClassifier,)
from src.models.random_forest import (RandomForestIncidentClassifier,)
from src.models.svm_classifier import (SVMClassifier,)
from train import (
    prepare_training_data,
    train_model,
)


logger = logging.getLogger(__name__)


def build_models():
    """
    Create all candidate models for benchmarking.

    Returns
    -------
    dict
        Model name and model instance.
    """

    return {
        "Logistic Regression":
            LogisticIncidentClassifier(),

        "Naive Bayes":
            NaiveBayesClassifier(),

        "Linear SVM":
            SVMClassifier(),

        "Decision Tree":
            DecisionTreeIncidentClassifier(),

        "Random Forest":
            RandomForestIncidentClassifier(),
    }


def benchmark_models():
    """
    Train and benchmark all candidate models.
    """

    logger.info(
        "Starting multi-model benchmark..."
    )

    # 1. Prepare dataset
    data = prepare_training_data()

    # 2. Build candidate models
    models = build_models()

    results = []

    trained_models = {}

    # 3. Train and evaluate every model
    for name, model in models.items():

        logger.info(
            "Benchmarking %s...",
            name,
        )

        # Train through train.py
        trained_model, training_time = train_model(
            model,
            data,
        )

        trained_models[name] = trained_model

        # Validation predictions
        val_predictions = trained_model.predict(
            data["x_val"]
        )

        # Test predictions
        test_predictions = trained_model.predict(
            data["x_test"]
        )

        # Calculate test metrics
        accuracy = accuracy_score(
            data["y_test"],
            test_predictions,
        )

        precision = precision_score(
            data["y_test"],
            test_predictions,
            average="weighted",
            zero_division=0,
        )

        recall = recall_score(
            data["y_test"],
            test_predictions,
            average="weighted",
            zero_division=0,
        )

        f1 = f1_score(
            data["y_test"],
            test_predictions,
            average="weighted",
            zero_division=0,
        )

        results.append(
            {
                "Model": name,
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "F1 Score": f1,
                "Training Time (s)": training_time,
            }
        )

    # 4. Create benchmark table
    results_df = pd.DataFrame(results)

    # 5. Rank models by F1
    results_df = results_df.sort_values(
        by="F1 Score",
        ascending=False,
    ).reset_index(drop=True)

    results_df.insert(
        0,
        "Rank",
        range(
            1,
            len(results_df) + 1,
        ),
    )

    # 6. Display results
    print("\n")
    print("=" * 80)
    print("SIRA MODEL BENCHMARK")
    print("=" * 80)

    print(
        results_df.to_string(
            index=False
        )
    )

    # 7. Best model
    best_model_name = results_df.iloc[0]["Model"]

    print("\n")
    print("=" * 80)
    print("BEST MODEL")
    print("=" * 80)

    print(
        f"Model: {best_model_name}"
    )

    print(
        f"F1 Score: "
        f"{results_df.iloc[0]['F1 Score']:.4f}"
    )

    logger.info(
        "Benchmark completed."
    )

    return {
        "results": results_df,
        "models": trained_models,
        "data": data,
        "best_model": best_model_name,
    }


if __name__ == "__main__":

    logging.basicConfig(
        level=logging.INFO,
        format="%(levelname)s: %(message)s",
    )

    benchmark_models()
```

> Before we proceed make sure to update the `evaluate.py` and `train.py` script.

`evaluate.py`:

```python
#evaluate.py
"""Detailed model evaluation for SIRA."""

import logging
import time

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


logger = logging.getLogger(__name__)


def evaluate_model(
    classifier,
    features,
    labels,
    dataset_name="Test",
    training_time=None,
):
    """
    Perform detailed evaluation of a trained model.

    Parameters
    ----------
    classifier : object
        Trained classifier.

    features : array-like
        Feature matrix.

    labels : array-like
        True labels.

    dataset_name : str
        Name of the dataset being evaluated.

    training_time : float, optional
        Model training time in seconds.

    Returns
    -------
    dict
        Complete evaluation results.
    """

    logger.info(
        "Evaluating model on %s dataset...",
        dataset_name,
    )

    # 1. Measure inference time
    start_time = time.perf_counter()

    predictions = classifier.predict(
        features
    )

    inference_time = (
        time.perf_counter() - start_time
    )

    # 2. Calculate metrics
    accuracy = accuracy_score(
        labels,
        predictions,
    )

    precision = precision_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    recall = recall_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    # 3. Classification report
    report = classification_report(
        labels,
        predictions,
        zero_division=0,
    )

    # 4. Confusion matrix
    matrix = confusion_matrix(
        labels,
        predictions,
    )

    # 5. Build result
    results = {
        "dataset": dataset_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "classification_report": report,
        "confusion_matrix": matrix,
        "training_time_seconds": training_time,
        "inference_time_seconds": inference_time,
    }

    return results


def print_evaluation_report(
    model_name,
    results,
):
    """
    Print a detailed evaluation report.
    """

    print("\n")
    print("=" * 80)
    print(
        f"{model_name.upper()} "
        f"- {results['dataset'].upper()} EVALUATION"
    )
    print("=" * 80)

    print(
        f"Accuracy  : "
        f"{results['accuracy']:.4f}"
    )

    print(
        f"Precision : "
        f"{results['precision']:.4f}"
    )

    print(
        f"Recall    : "
        f"{results['recall']:.4f}"
    )

    print(
        f"F1 Score  : "
        f"{results['f1_score']:.4f}"
    )

    if results["training_time_seconds"] is not None:

        print(
            f"Training Time : "
            f"{results['training_time_seconds']:.4f} seconds"
        )

    print(
        f"Inference Time: "
        f"{results['inference_time_seconds']:.6f} seconds"
    )

    print("\n")
    print("Classification Report")
    print("-" * 80)

    print(
        results["classification_report"]
    )

    print("Confusion Matrix")
    print("-" * 80)

    print(
        results["confusion_matrix"]
    )


def evaluate_models(
    models,
    features,
    labels,
    dataset_name="Test",
    training_times=None,
):
    """
    Evaluate multiple trained models.

    Returns
    -------
    tuple
        Detailed evaluation results and
        summary DataFrame.
    """

    if training_times is None:
        training_times = {}

    detailed_results = {}
    summary = []

    for model_name, classifier in models.items():

        results = evaluate_model(
            classifier=classifier,
            features=features,
            labels=labels,
            dataset_name=dataset_name,
            training_time=training_times.get(
                model_name
            ),
        )

        detailed_results[model_name] = results

        summary.append(
            {
                "Model": model_name,
                "Accuracy": results["accuracy"],
                "Precision": results["precision"],
                "Recall": results["recall"],
                "F1 Score": results["f1_score"],
                "Training Time (s)": (
                    results[
                        "training_time_seconds"
                    ]
                ),
                "Inference Time (s)": (
                    results[
                        "inference_time_seconds"
                    ]
                ),
            }
        )

    summary_df = pd.DataFrame(
        summary
    )

    summary_df = summary_df.sort_values(
        by="F1 Score",
        ascending=False,
    ).reset_index(drop=True)

    summary_df.insert(
        0,
        "Rank",
        range(
            1,
            len(summary_df) + 1,
        ),
    )

    return detailed_results, summary_df


def evaluate_pipeline(
    benchmark_output,
):
    """
    Evaluate trained benchmark models
    on the test dataset.
    """

    models = benchmark_output["models"]
    data = benchmark_output["data"]

    # Evaluate all models
    detailed_results, summary_df = evaluate_models(
        models=models,
        features=data["x_test"],
        labels=data["y_test"],
        dataset_name="Test",
    )

    # Display benchmark
    print("\n")
    print("=" * 80)
    print("FINAL MODEL EVALUATION")
    print("=" * 80)

    print(
        summary_df.to_string(
            index=False
        )
    )

    # Display detailed reports
    for model_name, results in detailed_results.items():

        print_evaluation_report(
            model_name,
            results,
        )

    # Best model
    best_model = summary_df.iloc[0]["Model"]

    print("\n")
    print("=" * 80)
    print("BEST MODEL")
    print("=" * 80)

    print(
        f"Model: {best_model}"
    )

    print(
        f"F1 Score: "
        f"{summary_df.iloc[0]['F1 Score']:.4f}"
    )

    return {
        "summary": summary_df,
        "details": detailed_results,
        "best_model": best_model,
    }
    
```

---

`train.py`:

```python
"""Training pipeline for SIRA."""

import logging
import time

from src.data_loader import DataLoader
from src.preprocessing import TextPreprocessor
from src.data_splitter import DataSplitter
from src.feature_engineering import FeatureEngineer


logger = logging.getLogger(__name__)


def prepare_training_data():
    """
    Load, preprocess, split, and transform the dataset.

    Returns
    -------
    dict
        Training, validation, and test data.
    """

    logger.info("Loading dataset...")

    loader = DataLoader()
    processor = TextPreprocessor()

    splitter = DataSplitter(
        test_size=0.2,
        validation_size=0.1,
        random_state=42,
    )

    engineer = FeatureEngineer(
        method="tfidf",
        max_features=5000,
    )

    # Load
    df = loader.load_data()

    logger.info(
        "Raw data shape: %s",
        df.shape,
    )

    # Preprocess
    processed_df = processor.preprocess_dataset(df)

    logger.info(
        "Processed data shape: %s",
        processed_df.shape,
    )

    # Split
    (
        x_train,
        x_val,
        x_test,
        y_train,
        y_val,
        y_test,
    ) = splitter.split(
        processed_df["report_text"],
        processed_df["incident_type"],
    )

    # Feature engineering
    x_train_features = engineer.fit_transform(
        x_train
    )

    x_val_features = engineer.transform(
        x_val
    )

    x_test_features = engineer.transform(
        x_test
    )

    logger.info(
        "Training features: %s",
        x_train_features.shape,
    )

    logger.info(
        "Validation features: %s",
        x_val_features.shape,
    )

    logger.info(
        "Test features: %s",
        x_test_features.shape,
    )

    return {
        "x_train": x_train_features,
        "y_train": y_train,

        "x_val": x_val_features,
        "y_val": y_val,

        "x_test": x_test_features,
        "y_test": y_test,

        "engineer": engineer,
        "processor": processor,
    }


def train_model(model, data):
    """
    Train a single model using prepared training data.

    Parameters
    ----------
    model : object
        Model implementing train() and predict().

    data : dict
        Prepared dataset.

    Returns
    -------
    tuple
        Trained model and training time.
    """

    model_name = model.__class__.__name__

    logger.info(
        "Training %s...",
        model_name,
    )

    start_time = time.perf_counter()

    model.train(
        data["x_train"],
        data["y_train"],
    )

    training_time = (
        time.perf_counter() - start_time
    )

    logger.info(
        "%s training completed in %.4f seconds.",
        model_name,
        training_time,
    )

    return model, training_time

```

### Imports

In [1]:
'''Imports setup'''
#!/usr/bin/env python3
import sys
import logging
from benchmark import benchmark_models


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

# Automatically finds the project root 'sira' and adds it to Python's path
PROJECT_ROOT = r"c:\Users\M.faisal\Desktop\sira"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

**Run it**

In [2]:
benchmark_models()

INFO: Starting multi-model benchmark...
INFO: Loading dataset...
INFO: Loading dataset...
INFO: 
Data Loaded Succesfully...
INFO: Raw data shape: (924, 10)
INFO: Cleaning started...
INFO: Removing duplicates...
INFO: Cleaning empty report...
INFO: Filling missing values...
INFO: Standardizing departments...
INFO: Standardizing severity...
INFO: Standardizing shifts...
INFO: Standardizing status...
INFO: Standardizing dates...
INFO: Cleaning report text...
INFO: 
Dataset cleaned successfully!
INFO: Processed data shape: (924, 10)
INFO: Dataset splitting completed.
INFO: Training samples: 646
INFO: Validation samples: 92
INFO: Test samples: 186
INFO: Fitting tfidf vectorizer on training data...
INFO: Feature engineering completed. Shape: (646, 70)
INFO: Data transformed. Shape: (92, 70)
INFO: Data transformed. Shape: (186, 70)
INFO: Training features: (646, 70)
INFO: Validation features: (92, 70)
INFO: Test features: (186, 70)
INFO: Benchmarking Logistic Regression...
INFO: Training Logi



SIRA MODEL BENCHMARK
 Rank               Model  Accuracy  Precision  Recall  F1 Score  Training Time (s)
    1 Logistic Regression       1.0        1.0     1.0       1.0           0.043389
    2         Naive Bayes       1.0        1.0     1.0       1.0           0.006835
    3          Linear SVM       1.0        1.0     1.0       1.0           0.024352
    4       Decision Tree       1.0        1.0     1.0       1.0           0.016218
    5       Random Forest       1.0        1.0     1.0       1.0           0.128408


BEST MODEL
Model: Logistic Regression
F1 Score: 1.0000


{'results':    Rank                Model  Accuracy  Precision  Recall  F1 Score  \
 0     1  Logistic Regression       1.0        1.0     1.0       1.0   
 1     2          Naive Bayes       1.0        1.0     1.0       1.0   
 2     3           Linear SVM       1.0        1.0     1.0       1.0   
 3     4        Decision Tree       1.0        1.0     1.0       1.0   
 4     5        Random Forest       1.0        1.0     1.0       1.0   
 
    Training Time (s)  
 0           0.043389  
 1           0.006835  
 2           0.024352  
 3           0.016218  
 4           0.128408  ,
 'models': {'Logistic Regression': <src.models.logistic_regression.LogisticIncidentClassifier at 0x1926ab57c10>,
  'Naive Bayes': <src.models.naive_bayes.NaiveBayesClassifier at 0x19269fae390>,
  'Linear SVM': <src.models.svm_classifier.SVMClassifier at 0x1926ab547d0>,
  'Decision Tree': <src.models.decision_tree.DecisionTreeIncidentClassifier at 0x1926ab54bd0>,
  'Random Forest': <src.models.random_forest.

<!-- From the output above you can see that all the models are having thesame score, which in our case its obvious since we're training using few datasets. Comparing `all` models, we took the first model i.e `Logistic Regression` as the best performing model. Now this gives you the understanding of why comparison matters. -->
The dataset size is one limitation, but we'll not conclude that the 1.0 scores are mainly because we only have 924 records. In fact, the bigger concern is that our dataset appears to be too easy/separable for the models, especially since all five fundamentally different algorithms achieved exactly 100%.

Our benchmark is actually useful because it has revealed something important about the dataset. What our result is telling us is that we have:

```text
924 records
646 training
92 validation
186 test
Only 70 TF-IDF features
5 very different classifiers
Every classifier: Accuracy = 1.00, Precision = 1.00, Recall = 1.00, F1 = 1.00
```

The probability that Logistic Regression, Naive Bayes, SVM, Decision Tree, and Random Forest would all perform identically at 100% on a genuinely difficult real-world incident classification problem is quite low.

So this is an investigation you can do before simply increasing the dataset size. Because our dataset may simply be too easy i.e for example, imagine the data contains reports like:
```text
"Employee slipped on wet floor in warehouse"
"Vehicle collision occurred at loading bay"
"Worker fell from scaffolding"
"Chemical spill detected in processing area"
"Fire outbreak reported in storage facility"
```

If the wording is highly consistent, TF-IDF can almost directly distinguish the classes. That could explain why even simple models achieve 100%. Real incident reports would usually be messier:

```text
"John was moving materials when he lost balance..."
"Small amount of liquid noticed around the pump..."
"Driver reported strange noise before impact..."
"Smoke observed around the electrical panel..."
```

The model then has to infer the incident type rather than simply recognize obvious keywords.

---

You can also run `main.py` with:

```python
"""Application entry point."""

from benchmark import benchmark_models

def main():
    """Run the SIRA machine learning pipeline."""
    
    results = benchmark_models()
    
    print(results)

if __name__ == "__main__":
    main()
    
```

---

**Discussion session** 

We want to be able to answer the much more meaningful question:

> **"Is SIRA actually learning to classify incident reports, or is our dataset making the task artificially easy?"**

And because we're building this as an **ML Engineering project**, Its strongly recommended that we perform this validation **before moving to model persistence, AWS deployment, or the Bedrock integration**.

The core pipeline is already working. The next task is to make sure the result is **trustworthy**, not merely that the code runs.

---


### Phase 4 deliverables

By the end of this phase, you should have:

* A modular project structure with one class per algorithm.
* Experience implementing five classical classifiers.
* A reusable `benchmark.py` script for evaluating models.
* A metrics table comparing all models.
* A visualization of model performance.
* An engineering report recommending the most suitable model.
* A solid understanding of why benchmarking is essential before deployment.

### Recommendation

As we have now identified the best-performing classical model, **Phase 5** should move beyond "training a model" to **building a production-ready inference pipeline**.

It should cover:

* Persisting the trained model and TF-IDF vectorizer with `joblib`.
* Creating a reusable prediction service (e.g., `PredictionService`).
* Loading artifacts without retraining.
* Handling unseen input safely.
* Designing a clean inference API that the future Streamlit app and REST API can both reuse.

This reinforces key lessons: **training is an offline process, while prediction is an online service**. Understanding that separation is fundamental to modern ML engineering.